# 03 — Data Cleaning and Integration
This notebook re-runs the harmonisation functions from `src/climatesense/data/harmonize.py` to
show each step (it does not overwrite the stored panel).

**Steps.** (1) schema normalisation and sentinel removal; (2) site definition
`(country, round(lat,0.1°), round(lon,0.1°))`; (3) same-day aggregation; (4) site selection
(≥ 500 observed days); (5) continuous daily grid; (6) spatial joins to World Cities and Berkeley
Earth by haversine distance:

$$ d = 2R\,\arcsin\sqrt{\sin^2\tfrac{\Delta\varphi}{2} + \cos\varphi_1\cos\varphi_2\sin^2\tfrac{\Delta\lambda}{2}},\quad R = 6371\ \text{km}.$$

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
from climatesense.utils import read_json
import platform, torch, sklearn
print("Executed:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "| Python", platform.python_version(),
      "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
def show(p, width=850):
    display(Image(filename=str(ROOT / p), width=width))

Executed: 2026-10-25 18:06 | Python 3.13.7 | torch 2.14.1+cu126 | sklearn 1.9.1 | pandas 3.1.0


In [2]:
from climatesense.data import harmonize as H
raw = H.load_weather_raw()
day, audit = H.clean_weather(raw)
print(json.dumps(audit, indent=1))

{
 "raw_rows": 170733,
 "invalid_values_set_nan": {
  "temp_c": 1,
  "feels_like_c": 1,
  "rh": 0,
  "wind_kph": 1,
  "gust_kph": 1,
  "pressure_hpa": 12,
  "precip_mm": 0,
  "cloud": 0,
  "visibility_km": 0,
  "uv_index": 0,
  "co": 1,
  "o3": 0,
  "no2": 0,
  "so2": 1,
  "pm25": 0,
  "pm10": 2,
  "us_epa_index": 0,
  "gb_defra_index": 0
 },
 "exact_duplicate_rows": 1,
 "same_day_records_averaged": 416
}


In [3]:
kept, stats = H.select_sites(day, 500)
print("sites found:", len(stats), "| kept (>=500 observed days):", stats.kept.sum(), "| site-days kept:", len(kept))
stats.sort_values("n_days").head(8)

sites found: 279 | kept (>=500 observed days): 185 | site-days kept: 160497


,site_id,country,location,lat,lon,n_days,first,last,span_days,kept
278,火鸡|38.8|34.6,火鸡,-Kingdom,38.8500,34.650,1,2024-07-13,2024-07-13,1,False
277,كولومبيا|6.4|-70.9,كولومبيا,Costa Rica,6.4300,-70.920,1,2024-07-13,2024-07-13,1,False
249,USA United States of America|39.5|-111.6,USA United States of America,Moroni,39.5300,-111.590,1,2024-06-25,2024-06-25,1,False
248,USA United States of America|33.8|-89.8,USA United States of America,Grenada,33.7800,-89.810,1,2024-06-25,2024-06-25,1,False
246,Turkménistan|37.7|65.4,Turkménistan,Krasnyy Turkmenistan,37.7000,65.370,1,2024-07-13,2024-07-13,1,False
256,United States of America|39.5|-111.6,United States of America,Moroni,39.5268,-111.585,1,2026-06-04,2026-06-04,1,False
255,United States of America|33.8|-89.8,United States of America,Grenada,33.7832,-89.807,1,2026-06-04,2026-06-04,1,False
250,USA United States of America|7.5|134.6,USA United States of America,Palau,7.5000,134.570,1,2024-06-25,2024-06-25,1,False


Examples of excluded fragments — provider substitutions and one-off multilingual records:

In [4]:
stats[~stats.kept].sort_values("n_days", ascending=False).head(10)[["country","location","n_days","first","last"]]

,country,location,n_days,first,last
169,Niger,Niamey,459,2024-05-16,2025-08-20
137,Madagascar,Ivory,457,2024-05-16,2025-08-20
113,Jamaica,Norman Gardens,453,2024-05-16,2026-10-12
21,Belgium,'S Gravenjansdijk,443,2024-05-31,2025-08-20
57,Costa Rica,San Ignacio,439,2024-05-31,2025-08-20
45,Cameroon,Ngaoundere,424,2024-06-20,2025-08-20
96,Iceland,Vestmannaeyjar,424,2024-06-20,2025-08-20
188,Paraguay,Aurora,423,2024-06-20,2025-08-20
112,Jamaica,Port Royal,420,2024-06-20,2025-08-20
170,Niger,Niamey,417,2025-08-21,2026-10-12


### Spatial joins (coordinates first, names only as a check)

In [5]:
sites = pd.read_csv(ROOT / "data/processed/sites.csv")
a = read_json(ROOT / "data/manifests/data_audit.json")
display(pd.DataFrame([a["world_cities_join"], a["berkeley_join"]], index=["World Cities", "Berkeley Earth"]))
sites.loc[sites.wc_matched & ~sites.wc_country_agrees, ["country","location","wc_city","wc_country","wc_distance_km"]]

,matched,unmatched,ambiguous_lt2km,country_name_disagreements,median_distance_km,berkeley_cities_with_normals
World Cities,182,3,40.0,9.0,1.317921,NaN
Berkeley Earth,156,29,NaN,NaN,73.035956,3510.0


,country,location,wc_city,wc_country,wc_distance_km
32,Cape Verde,Praia,Praia,Cabo Verde,1.540200
43,Democratic Republic of Congo,Kinshasa,Lingwala,Congo (Kinshasa),2.319652
55,Fiji Islands,Suva,Suva,Fiji,1.700215
87,Kyrghyzstan,Bishkek,Bishkek,Kyrgyzstan,2.812392
95,Macedonia,Skopje,Skopje,North Macedonia,0.442818
112,Myanmar,Rangoon,Rangoon,Burma,2.774961
119,North Korea,Pyongyang,Pyongyang,"Korea, North",0.639247
149,South Korea,Seoul,Seoul,"Korea, South",1.156346
155,Swaziland,Mbabane,Mbabane,Eswatini,0.090740


Name disagreements are naming variants (Swaziland/Eswatini, Myanmar/Burma …), confirming the joins.

In [6]:
sites.loc[~sites.wc_matched, ["country","location","lat","lon","wc_distance_km"]]

,country,location,lat,lon,wc_distance_km
20,Bolivia,National,-11.182780,-67.082780,96.566408
24,Brazil,Bras,-2.082780,-58.167220,63.581539
85,Kiribati,Tarawa,-0.882786,169.532786,292.786409


### Temporal alignment and leakage-safe preprocessing
* The grid is daily and continuous per site; unobserved days have `observed = 0`.
* Model inputs are forward-filled for **at most 3 days** (past information only); labels use the
  raw `_obs` columns and are never imputed.
* Berkeley monthly normals (1961–1990) are static climatology — no look-ahead is possible.
* Standardisation statistics are fitted on **training-period rows only** (`PanelArrays`).
* Chronological split with identical calendar cut-offs for all sites; a sample is used only if its
  first input day *and* its target day fall inside the same split.

In [7]:
panel = pd.read_parquet(ROOT / "data/processed/panel.parquet")
sp = read_json(ROOT / "data/manifests/splits.json"); print(sp)
from climatesense.training.trainer import get_arrays
arr = get_arrays()
for s in ["train","val","test"]:
    ix = arr.sample_index(14, 1, s)
    print(f"{s:5s} samples={len(ix):6d}  first input={arr.date_ns[ix-13].min()}  last target={arr.date_ns[ix+1].max()}")

{'train': ['2024-05-16', '2026-01-21'], 'val': ['2026-01-22', '2026-06-02'], 'test': ['2026-06-03', '2026-10-12']}


train samples=109762  first input=2024-05-16  last target=2026-01-21
val   samples= 21484  first input=2026-01-22  last target=2026-06-02
test  samples= 21192  first input=2026-06-03  last target=2026-10-12


The gap between consecutive splits (≥ 14 days) is the implicit embargo created by the window rule.